### Setup

In [ ]:
!conda create -n segformer python=3.9 -y

In [ ]:
!conda run -n segformer python -m pip install --upgrade pip setuptools wheel

!conda run -n segformer python -m pip install torch torchvision \
  --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!conda run -n segformer python -m pip install transformers accelerate pillow opencv-python numpy tqdm

In [ ]:
!conda run -n segformer python - <<'PY'
import torch
import transformers
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
PY

In [ ]:
!conda run -n segformer python -m pip show transformers

In [ ]:
!conda run -n segformer python -c "import torch, transformers; print('torch:', torch.__version__); print('cuda:', torch.cuda.is_available()); print('transformers:', transformers.__version__)"

In [ ]:
!mkdir -p /path/to/project/segmentation_scripts
!conda run -n segformer python -m pip install \
  safetensors huggingface_hub scipy matplotlib einops timm sentencepiece protobuf accelerate
!conda run -n segformer python -c "import torch, transformers, timm, einops; print('torch:', torch.__version__); print('cuda:', torch.cuda.is_available()); print('transformers:', transformers.__version__); print('imports OK')"

In [ ]:
%%writefile /path/to/project/segmentation_scripts/download_cityscapes_segmentors.py
import os
from pathlib import Path

os.environ["HF_HOME"] = "/path/to/project/hf_cache"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

from transformers import (
    AutoImageProcessor,
    SegformerForSemanticSegmentation,
    Mask2FormerForUniversalSegmentation,
    OneFormerProcessor,
    OneFormerForUniversalSegmentation,
)

models = [
    {
        "name": "SegFormer",
        "model_id": "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",
        "processor_cls": AutoImageProcessor,
        "model_cls": SegformerForSemanticSegmentation,
    },
    {
        "name": "Mask2Former",
        "model_id": "facebook/mask2former-swin-large-cityscapes-semantic",
        "processor_cls": AutoImageProcessor,
        "model_cls": Mask2FormerForUniversalSegmentation,
    },
    {
        "name": "OneFormer",
        "model_id": "shi-labs/oneformer_cityscapes_swin_large",
        "processor_cls": OneFormerProcessor,
        "model_cls": OneFormerForUniversalSegmentation,
    },
]

for item in models:
    print("=" * 80)
    print(f"Downloading/checking {item['name']}: {item['model_id']}")

    processor = item["processor_cls"].from_pretrained(item["model_id"])
    model = item["model_cls"].from_pretrained(item["model_id"])

    print(f"{item['name']} OK")

print("=" * 80)
print("All three segmentors downloaded/cached.")
print("HF cache:", os.environ["HF_HOME"])

In [ ]:
!conda run -n segformer python /path/to/project/segmentation_scripts/download_cityscapes_segmentors.py

In [ ]:
%%writefile /path/to/project/segmentation_scripts/run_cityscapes_segmentor.py
import argparse
import os
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoImageProcessor,
    SegformerForSemanticSegmentation,
    Mask2FormerForUniversalSegmentation,
    OneFormerProcessor,
    OneFormerForUniversalSegmentation,
)


os.environ["HF_HOME"] = "/path/to/project/hf_cache"


MODEL_CONFIGS = {
    "segformer": {
        "model_id": "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",
        "processor_cls": AutoImageProcessor,
        "model_cls": SegformerForSemanticSegmentation,
    },
    "mask2former": {
        "model_id": "facebook/mask2former-swin-large-cityscapes-semantic",
        "processor_cls": AutoImageProcessor,
        "model_cls": Mask2FormerForUniversalSegmentation,
    },
    "oneformer": {
        "model_id": "shi-labs/oneformer_cityscapes_swin_large",
        "processor_cls": OneFormerProcessor,
        "model_cls": OneFormerForUniversalSegmentation,
    },
}


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_type", required=True, choices=list(MODEL_CONFIGS.keys()))
    parser.add_argument("--input_dir", required=True)
    parser.add_argument("--output_dir", required=True)
    return parser.parse_args()


def list_images(input_dir):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    input_dir = Path(input_dir)
    return sorted([p for p in input_dir.iterdir() if p.suffix.lower() in exts])


def main():
    args = parse_args()

    input_dir = Path(args.input_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    cfg = MODEL_CONFIGS[args.model_type]
    model_id = cfg["model_id"]

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("=" * 80)
    print("Model type:", args.model_type)
    print("Model ID:", model_id)
    print("Input dir:", input_dir)
    print("Output dir:", output_dir)
    print("Device:", device)

    processor = cfg["processor_cls"].from_pretrained(model_id)
    model = cfg["model_cls"].from_pretrained(model_id)
    model.to(device)
    model.eval()

    image_paths = list_images(input_dir)
    print("Found images:", len(image_paths))

    for img_path in tqdm(image_paths):
        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        if args.model_type == "oneformer":
            inputs = processor(
                images=image,
                task_inputs=["semantic"],
                return_tensors="pt",
            )
        else:
            inputs = processor(images=image, return_tensors="pt")

        inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        if args.model_type == "segformer":
            logits = outputs.logits
            upsampled_logits = torch.nn.functional.interpolate(
                logits,
                size=(height, width),
                mode="bilinear",
                align_corners=False,
            )
            pred = upsampled_logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)

        else:
            semantic_maps = processor.post_process_semantic_segmentation(
                outputs,
                target_sizes=[(height, width)],
            )
            pred = semantic_maps[0].cpu().numpy().astype(np.uint8)

        out_path = output_dir / f"{img_path.stem}.png"
        Image.fromarray(pred).save(out_path)

    print("Done.")
    print("Saved masks to:", output_dir)


if __name__ == "__main__":
    main()

In [ ]:
!conda run -n segformer python /path/to/project/segmentation_scripts/run_cityscapes_segmentor.py \
  --model_type segformer \
  --input_dir /path/to/project/rain-rendering/data/output/diff_m_75_v1/leftImg8bit/train/aachen/rain/75mm/rainy_image \
  --output_dir /path/to/project/segmentation_outputs/segformer/rain_medium_1

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

mask_dir = Path("/path/to/project/segmentation_outputs/segformer/rain_medium_1")
masks = sorted(mask_dir.glob("*.png"))

print("num masks:", len(masks))

for p in masks[:5]:
    arr = np.array(Image.open(p))
    vals, counts = np.unique(arr, return_counts=True)
    print(p.name)
    print("shape:", arr.shape, "dtype:", arr.dtype)
    print("unique labels:", vals)
    print("counts:", dict(zip(vals.tolist(), counts.tolist())))
    print()

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

CITYSCAPES_PALETTE = np.array([
    [128,64,128],
    [244,35,232],
    [70,70,70],
    [102,102,156],
    [190,153,153],
    [153,153,153],
    [250,170,30],
    [220,220, 0],
    [107,142,35],
    [152,251,152],
    [ 70,130,180],
    [220, 20, 60],
    [255, 0, 0],
    [ 0, 0,142],
    [ 0, 0, 70],
    [ 0, 60,100],
    [ 0, 80,100],
    [ 0, 0,230],
    [119, 11, 32],
], dtype=np.uint8)

raw_dir = Path("/path/to/project/segmentation_outputs/segformer/rain_medium_1")
color_dir = Path("/path/to/project/segmentation_outputs_vis_color/segformer/rain_medium_1")
color_dir.mkdir(parents=True, exist_ok=True)

for p in sorted(raw_dir.glob("*.png")):
    mask = np.array(Image.open(p)).astype(np.uint8)
    color = CITYSCAPES_PALETTE[mask]
    Image.fromarray(color).save(color_dir / p.name)

print("Saved color visualizations to:", color_dir)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

img_path = sorted(color_dir.glob("*.png"))[0]
img = Image.open(img_path)

plt.figure(figsize=(12, 5))
plt.imshow(img)
plt.axis("off")
plt.show()

### segmenting

In [ ]:
from pathlib import Path

ROOT = Path("/path/to/project")

RAW_INPUTS = {
    "clean": ROOT / "cityscapes_subset_50/aachen",

    "rain_light_1": ROOT / "rain-rendering/data/output/diff_l_25_v1/leftImg8bit/train/aachen/rain/25mm/rainy_image",
    "rain_light_2": ROOT / "rain-rendering/data/output/diff_l_25_v2/leftImg8bit/train/aachen/rain/25mm/rainy_image",
    "rain_light_3": ROOT / "rain-rendering/data/output/diff_l_25_v3/leftImg8bit/train/aachen/rain/25mm/rainy_image",

    "rain_medium_1": ROOT / "rain-rendering/data/output/diff_m_75_v1/leftImg8bit/train/aachen/rain/75mm/rainy_image",
    "rain_medium_2": ROOT / "rain-rendering/data/output/diff_m_75_v2/leftImg8bit/train/aachen/rain/75mm/rainy_image",
    "rain_medium_3": ROOT / "rain-rendering/data/output/diff_m_75_v3/leftImg8bit/train/aachen/rain/75mm/rainy_image",

    "rain_heavy_1": ROOT / "rain-rendering/data/output/diff_h_140_v1/leftImg8bit/train/aachen/rain/140mm/rainy_image",
    "rain_heavy_2": ROOT / "rain-rendering/data/output/diff_h_140_v2/leftImg8bit/train/aachen/rain/140mm/rainy_image",
    "rain_heavy_3": ROOT / "rain-rendering/data/output/diff_h_140_v3/leftImg8bit/train/aachen/rain/140mm/rainy_image",
}

RAIN_DATASET_NAMES = [k for k in RAW_INPUTS if k != "clean"]
def image_count(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]
    return sum(len(list(folder.glob(ext))) for ext in exts)

In [ ]:
RESTORMER_NAME_MAP = {
    "rain_light_1": "light_v1",
    "rain_light_2": "light_v2",
    "rain_light_3": "light_v3",

    "rain_medium_1": "medium_v1",
    "rain_medium_2": "medium_v2",
    "rain_medium_3": "medium_v3",

    "rain_heavy_1": "heavy_v1",
    "rain_heavy_2": "heavy_v2",
    "rain_heavy_3": "heavy_v3",
}

SEG_INPUTS = {}
for name, folder in RAW_INPUTS.items():
    count = image_count(folder)
    if count > 0:
        SEG_INPUTS[name] = Path(folder)
    else:
        print("Missing/empty raw input:", name, folder)
for ds_name in RAIN_DATASET_NAMES:
    folder = ROOT / "derain_outputs/idt" / ds_name
    key = f"idt_{ds_name}"

    if image_count(folder) > 0:
        SEG_INPUTS[key] = folder
    else:
        print("Missing/empty IDT:", key, folder)
for ds_name in RAIN_DATASET_NAMES:
    folder = ROOT / "derain_outputs/drsformer" / ds_name / "Deraining"
    key = f"drsformer_{ds_name}"

    if image_count(folder) > 0:
        SEG_INPUTS[key] = folder
    else:
        print("Missing/empty DRSformer:", key, folder)
for ds_name in RAIN_DATASET_NAMES:
    folder = ROOT / "derain_outputs/nerdrain" / ds_name
    key = f"nerdrain_{ds_name}"

    if image_count(folder) > 0:
        SEG_INPUTS[key] = folder
    else:
        print("Missing/empty NeRD-Rain:", key, folder)
for ds_name in RAIN_DATASET_NAMES:
    restormer_name = RESTORMER_NAME_MAP[ds_name]
    folder = ROOT / "restormer_outputs" / restormer_name / "Deraining"
    key = f"restormer_{ds_name}"

    if image_count(folder) > 0:
        SEG_INPUTS[key] = folder
    else:
        print("Missing/empty Restormer:", key, folder)

print("=" * 100)
print("Total segmentation input folders:", len(SEG_INPUTS))
for key, folder in SEG_INPUTS.items():
    print(f"{key:35s} {image_count(folder):3d} {folder}")

In [ ]:
import subprocess
from pathlib import Path

SCRIPT = Path("/path/to/project/segmentation_scripts/run_cityscapes_segmentor.py")
OUT_BASE = Path("/path/to/project/segmentation_outputs")

HF_SEGMENTORS = [
    "segformer",
    "mask2former",
    "oneformer",
]

def mask_count(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return len(list(folder.glob("*.png")))

for segmentor in HF_SEGMENTORS:
    print("\n" + "#" * 100)
    print("STARTING SEGMENTOR:", segmentor)
    print("#" * 100)

    for input_name, input_dir in SEG_INPUTS.items():
        input_dir = Path(input_dir)
        expected = image_count(input_dir)

        out_dir = OUT_BASE / segmentor / input_name
        out_dir.mkdir(parents=True, exist_ok=True)

        existing = mask_count(out_dir)

        if expected > 0 and existing >= expected:
            print(f"SKIP {segmentor:12s} {input_name:35s} existing={existing} expected={expected}")
            continue

        print("=" * 100)
        print("Segmentor:", segmentor)
        print("Input set:", input_name)
        print("Input dir:", input_dir)
        print("Expected images:", expected)
        print("Output dir:", out_dir)

        cmd = [
            "conda", "run", "-n", "segformer", "python", str(SCRIPT),
            "--model_type", segmentor,
            "--input_dir", str(input_dir),
            "--output_dir", str(out_dir),
        ]

        result = subprocess.run(
            cmd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )

        print(result.stdout)

        final_count = mask_count(out_dir)
        print("Final mask count:", final_count, "Expected:", expected)

        if final_count != expected:
            print("WARNING: count mismatch:", segmentor, input_name, final_count, expected)

In [ ]:
from pathlib import Path
import pandas as pd

OUT_BASE = Path("/path/to/project/segmentation_outputs")

def mask_count(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return len(list(folder.glob("*.png")))

rows = []

for segmentor in ["segformer", "mask2former", "oneformer"]:
    for input_name, input_dir in SEG_INPUTS.items():
        expected = image_count(input_dir)
        out_dir = OUT_BASE / segmentor / input_name
        actual = mask_count(out_dir)

        rows.append({
            "segmentor": segmentor,
            "input_name": input_name,
            "expected": expected,
            "actual": actual,
            "done": actual >= expected,
            "output_dir": str(out_dir),
        })

status_df = pd.DataFrame(rows)

display(status_df.groupby("segmentor")["done"].agg(["sum", "count"]))
display(status_df[~status_df["done"]].head(30))

In [ ]:
from pathlib import Path
import pandas as pd

OUT_BASE = Path("/path/to/project/segmentation_outputs")

def mask_count(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return len(list(folder.glob("*.png")))

rows = []

for segmentor in ["segformer", "mask2former", "oneformer"]:
    for input_name, input_dir in SEG_INPUTS.items():
        expected = image_count(input_dir)
        out_dir = OUT_BASE / segmentor / input_name
        actual = mask_count(out_dir)

        rows.append({
            "segmentor": segmentor,
            "input_name": input_name,
            "expected": expected,
            "actual": actual,
            "done": actual >= expected,
            "output_dir": str(out_dir),
        })

status_df = pd.DataFrame(rows)

display(status_df.groupby("segmentor")["done"].agg(["sum", "count"]))
display(status_df[~status_df["done"]])

In [ ]:
from pathlib import Path
import pandas as pd

rows = []

for segmentor in ["segformer", "mask2former", "oneformer"]:
    for input_name, input_dir in SEG_INPUTS.items():
        expected = image_count(input_dir)
        out_dir = Path("/path/to/project/segmentation_outputs") / segmentor / input_name
        actual = mask_count(out_dir)

        rows.append({
            "segmentor": segmentor,
            "input_name": input_name,
            "expected": expected,
            "actual": actual,
            "ok": expected == actual,
            "output_dir": str(out_dir),
        })

seg_status_df = pd.DataFrame(rows)
seg_status_df.sort_values(["segmentor", "input_name"]).head()

In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/path/to/project")

UDR_NAME_MAP = {
    "rain_light_1": "light_v1",
    "rain_light_2": "light_v2",
    "rain_light_3": "light_v3",

    "rain_medium_1": "medium_v1",
    "rain_medium_2": "medium_v2",
    "rain_medium_3": "medium_v3",

    "rain_heavy_1": "heavy_v1",
    "rain_heavy_2": "heavy_v2",
    "rain_heavy_3": "heavy_v3",
}

UDR_SRC_BASE = ROOT / "udrs2former_outputs"
UDR_CLEAN_BASE = ROOT / "udrs2former_outputs_for_seg"

def list_image_files(folder):
    folder = Path(folder)
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])

UDR_SEG_INPUTS = {}

for ds_name, udr_folder_name in UDR_NAME_MAP.items():
    src_dir = UDR_SRC_BASE / udr_folder_name
    dst_dir = UDR_CLEAN_BASE / ds_name

    all_imgs = list_image_files(src_dir)
    non_cat_imgs = [p for p in all_imgs if not p.name.startswith("cat_")]

    print("=" * 80)
    print(ds_name)
    print("Source:", src_dir)
    print("All images:", len(all_imgs))
    print("Non-cat images:", len(non_cat_imgs))

    if len(non_cat_imgs) != 50:
        print("WARNING: expected 50 non-cat images, got", len(non_cat_imgs))
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for img_path in non_cat_imgs:
        link_path = dst_dir / img_path.name
        try:
            link_path.symlink_to(img_path)
        except FileExistsError:
            pass
        except OSError:
            shutil.copy2(img_path, link_path)

    UDR_SEG_INPUTS[f"udr_{ds_name}"] = dst_dir

print("=" * 80)
print("UDR segmentor input folders:")
for key, folder in UDR_SEG_INPUTS.items():
    print(f"{key:30s} {len(list_image_files(folder)):3d} {folder}")

In [ ]:
import subprocess
from pathlib import Path

SCRIPT = Path("/path/to/project/segmentation_scripts/run_cityscapes_segmentor.py")
OUT_BASE = Path("/path/to/project/segmentation_outputs")

UDR_SEGMENTORS = [
    "segformer",
    "mask2former",
    "oneformer",
]

def image_count(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]
    return sum(len(list(folder.glob(ext))) for ext in exts)

def mask_count(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return len(list(folder.glob("*.png")))

for segmentor in UDR_SEGMENTORS:
    print("\n" + "#" * 100)
    print("STARTING UDR SEGMENTOR:", segmentor)
    print("#" * 100)

    for input_name, input_dir in UDR_SEG_INPUTS.items():
        input_dir = Path(input_dir)
        expected = image_count(input_dir)

        out_dir = OUT_BASE / segmentor / input_name
        out_dir.mkdir(parents=True, exist_ok=True)

        existing = mask_count(out_dir)

        if expected > 0 and existing >= expected:
            print(f"SKIP {segmentor:12s} {input_name:35s} existing={existing} expected={expected}")
            continue

        print("=" * 100)
        print("Segmentor:", segmentor)
        print("Input set:", input_name)
        print("Input dir:", input_dir)
        print("Expected images:", expected)
        print("Output dir:", out_dir)

        cmd = [
            "conda", "run", "-n", "segformer", "python", str(SCRIPT),
            "--model_type", segmentor,
            "--input_dir", str(input_dir),
            "--output_dir", str(out_dir),
        ]

        result = subprocess.run(
            cmd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        print(result.stdout)

        final_count = mask_count(out_dir)
        print("Final mask count:", final_count, "Expected:", expected)

        if final_count != expected:
            print("WARNING: count mismatch:", segmentor, input_name, final_count, expected)

In [ ]:
import pandas as pd
from pathlib import Path

rows = []

for segmentor in UDR_SEGMENTORS:
    for input_name, input_dir in UDR_SEG_INPUTS.items():
        expected = image_count(input_dir)
        out_dir = Path("/path/to/project/segmentation_outputs") / segmentor / input_name
        actual = mask_count(out_dir)

        rows.append({
            "segmentor": segmentor,
            "input_name": input_name,
            "expected": expected,
            "actual": actual,
            "ok": expected == actual,
            "output_dir": str(out_dir),
        })

udr_seg_status_df = pd.DataFrame(rows)

print("Failures:")
display(udr_seg_status_df[~udr_seg_status_df["ok"]])

print("Summary:")
display(udr_seg_status_df)

### output path

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/path/to/project")
SEG_OUT_BASE = ROOT / "segmentation_outputs"

SEGMENTORS = [
    "mseg",
    "segformer",
    "mask2former",
    "oneformer",
]

def count_masks(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0

    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]
    return sum(len(list(folder.glob(ext))) for ext in exts)

def count_input_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0

    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]
    return sum(len(list(folder.glob(ext))) for ext in exts)

rows = []
if "SEG_INPUTS" in globals():
    input_items = list(SEG_INPUTS.items())

    for segmentor in SEGMENTORS:
        for input_name, input_dir in input_items:
            input_dir = Path(input_dir)
            expected = count_input_images(input_dir)

            out_dir = SEG_OUT_BASE / segmentor / input_name
            actual = count_masks(out_dir)

            rows.append({
                "segmentor": segmentor,
                "input_name": input_name,
                "expected_images": expected,
                "actual_masks": actual,
                "complete": actual >= expected and expected > 0,
                "input_dir": str(input_dir),
                "output_dir": str(out_dir),
            })

else:
    print("SEG_INPUTS not found in memory, scanning existing output folders only.")
    for segmentor in SEGMENTORS:
        seg_dir = SEG_OUT_BASE / segmentor

        if not seg_dir.exists():
            rows.append({
                "segmentor": segmentor,
                "input_name": None,
                "expected_images": None,
                "actual_masks": 0,
                "complete": False,
                "input_dir": None,
                "output_dir": str(seg_dir),
            })
            continue

        for out_dir in sorted([p for p in seg_dir.iterdir() if p.is_dir()]):
            rows.append({
                "segmentor": segmentor,
                "input_name": out_dir.name,
                "expected_images": None,
                "actual_masks": count_masks(out_dir),
                "complete": None,
                "input_dir": None,
                "output_dir": str(out_dir),
            })

seg_outputs_df = pd.DataFrame(rows)
seg_outputs_df = seg_outputs_df.sort_values(
    ["segmentor", "input_name"],
    na_position="last"
).reset_index(drop=True)

print("Total output folder rows:", len(seg_outputs_df))

display(seg_outputs_df)
csv_path = ROOT / "segmentation_outputs_manifest.csv"
seg_outputs_df.to_csv(csv_path, index=False)
print("Saved manifest to:", csv_path)

In [ ]:
summary_df = (
    seg_outputs_df
    .groupby("segmentor")
    .agg(
        output_folders=("output_dir", "count"),
        total_masks=("actual_masks", "sum"),
        complete_folders=("complete", lambda x: (x == True).sum()),
    )
    .reset_index()
)

display(summary_df)

In [ ]:
incomplete_df = seg_outputs_df[
    (seg_outputs_df["complete"] == False) |
    (seg_outputs_df["actual_masks"] == 0)
]

display(incomplete_df)

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

ROOT = Path("/path/to/project")
MSEG_STANDARD_BASE = ROOT / "segmentation_outputs/mseg"

def list_mask_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])

def canonicalize_mseg_key(key):
    """
    Convert MSeg keys like:
        drsformer_heavy_v1
        idt_light_v2
        udrs2former_medium_v3

    into standardized input names like:
        drsformer_rain_heavy_1
        idt_rain_light_2
        udr_rain_medium_3
    """
    parts = key.split("_")

    derainer = parts[0]
    severity = parts[1]
    version = parts[2]

    version_num = version.replace("v", "")

    if derainer == "udrs2former":
        derainer = "udr"

    return f"{derainer}_rain_{severity}_{version_num}"

rows = []

for original_key, rel_path in PRED_DIRS_ALL.items():
    src_dir = ROOT / rel_path
    canonical_name = canonicalize_mseg_key(original_key)
    dst_dir = MSEG_STANDARD_BASE / canonical_name

    all_masks = list_mask_files(src_dir)
    non_cat_masks = [p for p in all_masks if not p.name.startswith("cat_")]

    # Recreate standardized folder
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    # Symlink only non-cat masks
    for mask_path in non_cat_masks:
        link_path = dst_dir / mask_path.name
        try:
            link_path.symlink_to(mask_path)
        except OSError:
            shutil.copy2(mask_path, link_path)

    rows.append({
        "mseg_original_key": original_key,
        "canonical_input_name": canonical_name,
        "source_dir": str(src_dir),
        "standardized_dir": str(dst_dir),
        "source_exists": src_dir.exists(),
        "total_masks_source": len(all_masks),
        "non_cat_masks_source": len(non_cat_masks),
        "standardized_masks": len(list_mask_files(dst_dir)),
        "ok_50_noncat": len(non_cat_masks) == 50,
    })

mseg_standard_df = pd.DataFrame(rows).sort_values("canonical_input_name").reset_index(drop=True)

display(mseg_standard_df)

print("Failures / non-50 folders:")
display(mseg_standard_df[~mseg_standard_df["ok_50_noncat"]])

print("Saved standardized MSeg folders under:", MSEG_STANDARD_BASE)

remaining udr dataset

In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/path/to/project")

UDR_NAME_MAP = {
    "rain_light_1": "light_v1",
    "rain_light_2": "light_v2",
    "rain_light_3": "light_v3",

    "rain_medium_1": "medium_v1",
    "rain_medium_2": "medium_v2",
    "rain_medium_3": "medium_v3",

    "rain_heavy_1": "heavy_v1",
    "rain_heavy_2": "heavy_v2",
    "rain_heavy_3": "heavy_v3",
}

UDR_SRC_BASE = ROOT / "udrs2former_outputs"
UDR_CLEAN_BASE = ROOT / "udrs2former_outputs_for_seg"

def list_image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])

UDR_SEG_INPUTS = {}

for ds_name, udr_folder_name in UDR_NAME_MAP.items():
    src_dir = UDR_SRC_BASE / udr_folder_name
    dst_dir = UDR_CLEAN_BASE / ds_name

    all_imgs = list_image_files(src_dir)
    non_cat_imgs = [p for p in all_imgs if not p.name.startswith("cat_")]

    print("=" * 80)
    print(ds_name)
    print("Source:", src_dir)
    print("All images:", len(all_imgs))
    print("Non-cat images:", len(non_cat_imgs))

    if len(non_cat_imgs) != 50:
        print("WARNING: expected 50 non-cat images, got", len(non_cat_imgs))

    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    for img_path in non_cat_imgs:
        link_path = dst_dir / img_path.name
        try:
            link_path.symlink_to(img_path)
        except OSError:
            shutil.copy2(img_path, link_path)

    UDR_SEG_INPUTS[f"udr_{ds_name}"] = dst_dir

print("=" * 80)
print("UDR inputs prepared:")
for key, folder in UDR_SEG_INPUTS.items():
    print(f"{key:30s} {len(list_image_files(folder)):3d} {folder}")

In [ ]:
import subprocess
from pathlib import Path

SCRIPT = Path("/path/to/project/segmentation_scripts/run_cityscapes_segmentor.py")
OUT_BASE = Path("/path/to/project/segmentation_outputs")

UDR_SEGMENTORS = [
    "segformer",
    "mask2former",
    "oneformer",
]

def image_count(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]
    return sum(len(list(folder.glob(ext))) for ext in exts)

def mask_count(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return len(list(folder.glob("*.png")))

for segmentor in UDR_SEGMENTORS:
    print("\n" + "#" * 100)
    print("STARTING UDR SEGMENTOR:", segmentor)
    print("#" * 100)

    for input_name, input_dir in UDR_SEG_INPUTS.items():
        input_dir = Path(input_dir)
        expected = image_count(input_dir)

        out_dir = OUT_BASE / segmentor / input_name
        out_dir.mkdir(parents=True, exist_ok=True)

        existing = mask_count(out_dir)

        if expected > 0 and existing >= expected:
            print(f"SKIP {segmentor:12s} {input_name:35s} existing={existing} expected={expected}")
            continue

        print("=" * 100)
        print("Segmentor:", segmentor)
        print("Input set:", input_name)
        print("Input dir:", input_dir)
        print("Expected images:", expected)
        print("Existing masks:", existing)
        print("Output dir:", out_dir)

        cmd = [
            "conda", "run", "-n", "segformer", "python", str(SCRIPT),
            "--model_type", segmentor,
            "--input_dir", str(input_dir),
            "--output_dir", str(out_dir),
        ]

        result = subprocess.run(
            cmd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )

        print(result.stdout)

        final_count = mask_count(out_dir)
        print("Final mask count:", final_count, "Expected:", expected)

        if final_count != expected:
            print("WARNING: count mismatch:", segmentor, input_name, final_count, expected)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/path/to/project")
OUT_BASE = ROOT / "segmentation_outputs"

def count_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]
    return sum(len(list(folder.glob(ext))) for ext in exts)

udr_status_rows = []

for segmentor in ["segformer", "mask2former", "oneformer"]:
    for input_name, input_dir in UDR_SEG_INPUTS.items():
        out_dir = OUT_BASE / segmentor / input_name

        udr_status_rows.append({
            "segmentor": segmentor,
            "input_name": input_name,
            "input_dir": str(input_dir),
            "input_count": count_images(input_dir),
            "output_dir": str(out_dir),
            "output_count": count_images(out_dir),
            "ok": count_images(input_dir) == count_images(out_dir) == 50,
        })

udr_status_df = pd.DataFrame(udr_status_rows)
display(udr_status_df)

print("Failures:")
display(udr_status_df[~udr_status_df["ok"]])

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Pick one segmentor and one UDR dataset
segmentor = "segformer"   # options: segformer, mask2former, oneformer
udr_name = "udr_rain_light_1"

mask_dir = Path(f"/path/to/project/segmentation_outputs/{segmentor}/{udr_name}")
input_dir = Path(f"/path/to/project/udrs2former_outputs_for_seg/rain_light_1")

print("Mask dir:", mask_dir)
print("Input dir:", input_dir)
print("Mask count:", len(list(mask_dir.glob('*.png'))))
print("Input count:", len(list(input_dir.glob('*.png'))))

CITYSCAPES_PALETTE = np.array([
    [128, 64,128],   # 0 road
    [244, 35,232],   # 1 sidewalk
    [ 70, 70, 70],   # 2 building
    [102,102,156],   # 3 wall
    [190,153,153],   # 4 fence
    [153,153,153],   # 5 pole
    [250,170, 30],   # 6 traffic light
    [220,220,  0],   # 7 traffic sign
    [107,142, 35],   # 8 vegetation
    [152,251,152],   # 9 terrain
    [ 70,130,180],   # 10 sky
    [220, 20, 60],   # 11 person
    [255,  0,  0],   # 12 rider
    [  0,  0,142],   # 13 car
    [  0,  0, 70],   # 14 truck
    [  0, 60,100],   # 15 bus
    [  0, 80,100],   # 16 train
    [  0,  0,230],   # 17 motorcycle
    [119, 11, 32],   # 18 bicycle
], dtype=np.uint8)

mask_paths = sorted(mask_dir.glob("*.png"))
mask_path = mask_paths[0]

mask = np.array(Image.open(mask_path)).astype(np.uint8)
print("Example mask:", mask_path.name)
print("Unique labels:", np.unique(mask))

color_mask = CITYSCAPES_PALETTE[mask]

# Try to find matching UDR image
input_path = input_dir / mask_path.name
if not input_path.exists():
    print("Matching input image not found:", input_path)
    input_img = None
else:
    input_img = Image.open(input_path).convert("RGB")

plt.figure(figsize=(16, 6))

if input_img is not None:
    plt.subplot(1, 2, 1)
    plt.imshow(input_img)
    plt.title("UDR derained input")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(color_mask)
    plt.title(f"{segmentor} colored segmentation")
    plt.axis("off")
else:
    plt.imshow(color_mask)
    plt.title(f"{segmentor} colored segmentation")
    plt.axis("off")

plt.show()

### Plots and Graphs

#### Rain instance
MSeg

In [ ]:
PRED_DIRS_ALL = {
    "drsformer_heavy_v1": "mseg-semantic/temp_files/mseg-3m-480p_drsformer_heavy_v1_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "drsformer_heavy_v2": "mseg-semantic/temp_files/mseg-3m-480p_drsformer_heavy_v2_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "drsformer_heavy_v3": "mseg-semantic/temp_files/mseg-3m-480p_drsformer_heavy_v3_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "drsformer_light_v1": "mseg-semantic/temp_files/mseg-3m-480p_drsformer_light_v1_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "drsformer_light_v2": "mseg-semantic/temp_files/mseg-3m-480p_drsformer_light_v2_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "drsformer_light_v3": "mseg-semantic/temp_files/mseg-3m-480p_drsformer_light_v3_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "drsformer_medium_v1": "mseg-semantic/temp_files/mseg-3m-480p_drsformer_medium_v1_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "drsformer_medium_v2": "mseg-semantic/temp_files/mseg-3m-480p_drsformer_medium_v2_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "drsformer_medium_v3": "mseg-semantic/temp_files/mseg-3m-480p_drsformer_medium_v3_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "idt_heavy_v1": "mseg-semantic/temp_files/mseg-3m-480p_idt_heavy_v1_rain_heavy_1_universal_ss/360/gray",  # total=50, non-cat=50
    "idt_heavy_v2": "mseg-semantic/temp_files/mseg-3m-480p_idt_heavy_v2_rain_heavy_2_universal_ss/360/gray",  # total=50, non-cat=50
    "idt_heavy_v3": "mseg-semantic/temp_files/mseg-3m-480p_idt_heavy_v3_rain_heavy_3_universal_ss/360/gray",  # total=50, non-cat=50
    "idt_light_v1": "mseg-semantic/temp_files/mseg-3m-480p_idt_light_v1_rain_light_1_universal_ss/360/gray",  # total=50, non-cat=50
    "idt_light_v2": "mseg-semantic/temp_files/mseg-3m-480p_idt_light_v2_rain_light_2_universal_ss/360/gray",  # total=50, non-cat=50
    "idt_light_v3": "mseg-semantic/temp_files/mseg-3m-480p_idt_light_v3_rain_light_3_universal_ss/360/gray",  # total=50, non-cat=50
    "idt_medium_v1": "mseg-semantic/temp_files/mseg-3m-480p_idt_medium_v1_rain_medium_1_universal_ss/360/gray",  # total=50, non-cat=50
    "idt_medium_v2": "mseg-semantic/temp_files/mseg-3m-480p_idt_medium_v2_rain_medium_2_universal_ss/360/gray",  # total=50, non-cat=50
    "idt_medium_v3": "mseg-semantic/temp_files/mseg-3m-480p_idt_medium_v3_rain_medium_3_universal_ss/360/gray",  # total=50, non-cat=50
    "nerdrain_heavy_v1": "mseg-semantic/temp_files/mseg-3m-480p_nerdrain_heavy_v1_rain_heavy_1_universal_ss/360/gray",  # total=50, non-cat=50
    "nerdrain_heavy_v2": "mseg-semantic/temp_files/mseg-3m-480p_nerdrain_heavy_v2_rain_heavy_2_universal_ss/360/gray",  # total=50, non-cat=50
    "nerdrain_heavy_v3": "mseg-semantic/temp_files/mseg-3m-480p_nerdrain_heavy_v3_rain_heavy_3_universal_ss/360/gray",  # total=50, non-cat=50
    "nerdrain_light_v1": "mseg-semantic/temp_files/mseg-3m-480p_nerdrain_light_v1_rain_light_1_universal_ss/360/gray",  # total=50, non-cat=50
    "nerdrain_light_v2": "mseg-semantic/temp_files/mseg-3m-480p_nerdrain_light_v2_rain_light_2_universal_ss/360/gray",  # total=50, non-cat=50
    "nerdrain_light_v3": "mseg-semantic/temp_files/mseg-3m-480p_nerdrain_light_v3_rain_light_3_universal_ss/360/gray",  # total=50, non-cat=50
    "nerdrain_medium_v1": "mseg-semantic/temp_files/mseg-3m-480p_nerdrain_medium_v1_rain_medium_1_universal_ss/360/gray",  # total=50, non-cat=50
    "nerdrain_medium_v2": "mseg-semantic/temp_files/mseg-3m-480p_nerdrain_medium_v2_rain_medium_2_universal_ss/360/gray",  # total=50, non-cat=50
    "nerdrain_medium_v3": "mseg-semantic/temp_files/mseg-3m-480p_nerdrain_medium_v3_rain_medium_3_universal_ss/360/gray",  # total=50, non-cat=50
    "restormer_heavy_v1": "mseg-semantic/temp_files/mseg-3m-480p_restormer_heavy_v1_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "restormer_heavy_v2": "mseg-semantic/temp_files/mseg-3m-480p_restormer_heavy_v2_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "restormer_heavy_v3": "mseg-semantic/temp_files/mseg-3m-480p_restormer_heavy_v3_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "restormer_light_v1": "mseg-semantic/temp_files/mseg-3m-480p_restormer_light_v1_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "restormer_light_v2": "mseg-semantic/temp_files/mseg-3m-480p_restormer_light_v2_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "restormer_light_v3": "mseg-semantic/temp_files/mseg-3m-480p_restormer_light_v3_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "restormer_medium_v1": "mseg-semantic/temp_files/mseg-3m-480p_restormer_medium_v1_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "restormer_medium_v2": "mseg-semantic/temp_files/mseg-3m-480p_restormer_medium_v2_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "restormer_medium_v3": "mseg-semantic/temp_files/mseg-3m-480p_restormer_medium_v3_Deraining_universal_ss/360/gray",  # total=50, non-cat=50
    "udrs2former_heavy_v1": "mseg-semantic/temp_files/mseg-3m-480p_udrs2former_heavy_v1_heavy_v1_universal_ss/360/gray",  # total=100, non-cat=50
    "udrs2former_heavy_v2": "mseg-semantic/temp_files/mseg-3m-480p_udrs2former_heavy_v2_heavy_v2_universal_ss/360/gray",  # total=50, non-cat=50
    "udrs2former_heavy_v3": "mseg-semantic/temp_files/mseg-3m-480p_udrs2former_heavy_v3_heavy_v3_universal_ss/360/gray",  # total=50, non-cat=50
    "udrs2former_light_v1": "mseg-semantic/temp_files/mseg-3m-480p_udrs2former_light_v1_light_v1_universal_ss/360/gray",  # total=100, non-cat=50
    "udrs2former_light_v2": "mseg-semantic/temp_files/mseg-3m-480p_udrs2former_light_v2_light_v2_universal_ss/360/gray",  # total=50, non-cat=50
    "udrs2former_light_v3": "mseg-semantic/temp_files/mseg-3m-480p_udrs2former_light_v3_light_v3_universal_ss/360/gray",  # total=50, non-cat=50
    "udrs2former_medium_v1": "mseg-semantic/temp_files/mseg-3m-480p_udrs2former_medium_v1_medium_v1_universal_ss/360/gray",  # total=100, non-cat=50
    "udrs2former_medium_v2": "mseg-semantic/temp_files/mseg-3m-480p_udrs2former_medium_v2_medium_v2_universal_ss/360/gray",  # total=50, non-cat=50
    "udrs2former_medium_v3": "mseg-semantic/temp_files/mseg-3m-480p_udrs2former_medium_v3_medium_v3_universal_ss/360/gray",  # total=50, non-cat=50
}

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path("/path/to/project")

def resolve_path(path_str):
    p = Path(path_str)
    if p.exists():
        return p

    p2 = ROOT / path_str
    if p2.exists():
        return p2

    return p2
def list_masks(folder, derainer=None, max_non_cat=50):
    folder = Path(folder)
    if not folder.exists():
        return []

    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])
    files = [p for p in files if not p.name.startswith("cat_")]
    if derainer in {"udrs2former", "udr"}:
        files = files[:max_non_cat]

    return files

def parse_key(key):
    m = re.match(r"^(?P<derainer>.+)_(?P<severity>light|medium|heavy)_(?P<version>v[123])$", key)
    if m is None:
        raise ValueError(f"Could not parse key: {key}")
    return m.group("derainer"), m.group("severity"), m.group("version")

def read_mask(path):
    return np.array(Image.open(path))

def resize_mask_nearest(mask, target_shape):
    if mask.shape[:2] == target_shape:
        return mask
    img = Image.fromarray(mask.astype(np.uint8))
    img = img.resize((target_shape[1], target_shape[0]), resample=Image.NEAREST)
    return np.array(img)

def mask_miou(mask_a, mask_b, ignore_labels=(255,)):
    if mask_b.shape[:2] != mask_a.shape[:2]:
        mask_b = resize_mask_nearest(mask_b, mask_a.shape[:2])
    labels = np.union1d(np.unique(mask_a), np.unique(mask_b))
    labels = [int(x) for x in labels if int(x) not in ignore_labels]
    ious = []
    for label in labels:
        a = mask_a == label
        b = mask_b == label
        union = np.logical_or(a, b).sum()
        if union == 0:
            continue
        inter = np.logical_and(a, b).sum()
        ious.append(inter / union)
    return float(np.mean(ious)) if ious else np.nan

In [ ]:
rows = []

for key, path_str in PRED_DIRS_ALL.items():
    derainer, severity, version = parse_key(key)
    folder = resolve_path(path_str)
    masks = list_masks(folder, derainer=derainer)

    rows.append({
        "key": key,
        "derainer": derainer,
        "severity": severity,
        "version": version,
        "folder": str(folder),
        "exists": folder.exists(),
        "non_cat_count_used": len(masks),
    })

mseg_pred_status_df = pd.DataFrame(rows).sort_values(
    ["derainer", "severity", "version"]
).reset_index(drop=True)

display(mseg_pred_status_df)

print("Missing folders:")
display(mseg_pred_status_df[~mseg_pred_status_df["exists"]])

print("Non-50 folders:")
display(mseg_pred_status_df[mseg_pred_status_df["non_cat_count_used"] != 50])

In [ ]:
from pathlib import Path
import re
import numpy as np
from PIL import Image

ROOT = Path("/path/to/project")

def resolve_path(path_str):
    p = Path(path_str)
    if p.exists():
        return p
    p2 = ROOT / path_str
    return p2

def list_masks(folder, derainer=None, max_non_cat=50):
    folder = Path(folder)
    if not folder.exists():
        return []
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])
    files = [p for p in files if not p.name.startswith("cat_")]
    if derainer in {"udrs2former", "udr"}:
        files = files[:max_non_cat]
    return files

def parse_key(key):
    m = re.match(
        r"^(?P<derainer>.+)_(?P<severity>light|medium|heavy)_(?P<version>v[123])$",
        key
    )
    if m is None:
        raise ValueError(f"Could not parse key: {key}")
    return m.group("derainer"), m.group("severity"), m.group("version")
mask_maps = {}

for key, rel_path in PRED_DIRS_ALL.items():
    derainer, severity, version = parse_key(key)
    folder = resolve_path(rel_path)
    masks = list_masks(folder, derainer=derainer)
    mask_maps[(derainer, severity, version)] = {
        p.name: p for p in masks
    }
print("Built mask_maps groups:", len(mask_maps))

for group_key, file_map in sorted(mask_maps.items()):
    print(group_key, len(file_map))

In [ ]:
def read_mask(path):
    return np.array(Image.open(path))
def resize_mask_nearest(mask, target_shape):
    if mask.shape[:2] == target_shape:
        return mask
    img = Image.fromarray(mask.astype(np.uint8))
    img = img.resize((target_shape[1], target_shape[0]), resample=Image.NEAREST)
    return np.array(img)
def mask_miou(mask_a, mask_b, ignore_labels=(255,)):
    if mask_b.shape[:2] != mask_a.shape[:2]:
        mask_b = resize_mask_nearest(mask_b, mask_a.shape[:2])
    labels = np.union1d(np.unique(mask_a), np.unique(mask_b))
    labels = [int(x) for x in labels if int(x) not in ignore_labels]
    ious = []
    for label in labels:
        a = mask_a == label
        b = mask_b == label
        union = np.logical_or(a, b).sum()
        if union == 0:
            continue
        inter = np.logical_and(a, b).sum()
        ious.append(inter / union)

    return float(np.mean(ious)) if ious else np.nan

In [ ]:
severity_order = ["light", "medium", "heavy"]
version_order = ["v1", "v2", "v3"]

pair_rows = []

derainers = sorted({parse_key(k)[0] for k in PRED_DIRS_ALL.keys()})

for derainer in derainers:
    for severity in severity_order:
        needed = [(derainer, severity, v) for v in version_order]
        if not all(k in mask_maps for k in needed):
            print("Skipping incomplete group:", derainer, severity)
            continue
        maps = {v: mask_maps[(derainer, severity, v)] for v in version_order}
        common_names = set(maps["v1"].keys()) & set(maps["v2"].keys()) & set(maps["v3"].keys())
        common_names = sorted(common_names)
        print(f"{derainer:15s} {severity:7s} common images: {len(common_names)}")
        for image_name in common_names:
            masks = {
                v: read_mask(maps[v][image_name])
                for v in version_order
            }
            miou_12 = mask_miou(masks["v1"], masks["v2"])
            miou_13 = mask_miou(masks["v1"], masks["v3"])
            miou_23 = mask_miou(masks["v2"], masks["v3"])
            version_scores = {
                "v1": np.nanmean([miou_12, miou_13]),
                "v2": np.nanmean([miou_12, miou_23]),
                "v3": np.nanmean([miou_13, miou_23]),
            }
            for version, score in version_scores.items():
                pair_rows.append({
                    "segmentor": "mseg",
                    "derainer": derainer,
                    "severity": severity,
                    "version": version,
                    "image_name": image_name,
                    "miou_to_other_rain_instances": score,
                    "miou_v1_v2": miou_12,
                    "miou_v1_v3": miou_13,
                    "miou_v2_v3": miou_23,
                })
mseg_rain_instance_df = pd.DataFrame(pair_rows)
print("Total rows:", len(mseg_rain_instance_df))
display(mseg_rain_instance_df.head())
print("Rows per group:")
display(
    mseg_rain_instance_df
    .groupby(["derainer", "severity", "version"])
    .size()
    .reset_index(name="n")
)

In [ ]:
mseg_rain_instance_summary_df = (
    mseg_rain_instance_df
    .groupby(["segmentor", "derainer", "severity", "version"])
    .agg(
        n=("miou_to_other_rain_instances", "count"),
        mean_miou=("miou_to_other_rain_instances", "mean"),
        median_miou=("miou_to_other_rain_instances", "median"),
        std_miou=("miou_to_other_rain_instances", "std"),
        min_miou=("miou_to_other_rain_instances", "min"),
        max_miou=("miou_to_other_rain_instances", "max"),
    )
    .reset_index()
    .sort_values(["derainer", "severity", "version"])
)

display(mseg_rain_instance_summary_df)

out_csv = ROOT / "analysis_outputs/mseg_rain_instance_miou_summary.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
mseg_rain_instance_summary_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
PLOT_DIR = ROOT / "analysis_outputs/mseg_rain_instance_boxplots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

severity_order = ["light", "medium", "heavy"]
version_order = ["v1", "v2", "v3"]

for derainer in sorted(mseg_rain_instance_df["derainer"].unique()):
    df_d = mseg_rain_instance_df[mseg_rain_instance_df["derainer"] == derainer]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    for ax, severity in zip(axes, severity_order):
        df_s = df_d[df_d["severity"] == severity]
        data = [
            df_s[df_s["version"] == version]["miou_to_other_rain_instances"].dropna().values
            for version in version_order
        ]
        ax.boxplot(data, labels=version_order, showmeans=True)
        ax.set_title(f"{severity.capitalize()} rain")
        ax.set_xlabel("Rain instance version")
        ax.grid(axis="y", alpha=0.3)
        counts = [len(x) for x in data]
        ax.text(
            0.5,
            -0.18,
            f"n={counts}",
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=9,
        )
    axes[0].set_ylabel("mIoU agreement with other rain instances")
    fig.suptitle(f"MSeg rain-instance stability | Derainer: {derainer}", y=1.03)
    fig.tight_layout()
    out_path = PLOT_DIR / f"mseg_{derainer}_rain_instance_miou_boxplot.png"
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)

In [ ]:
derainer_order = sorted(mseg_rain_instance_df["derainer"].unique())
severity_order = ["light", "medium", "heavy"]
version_order = ["v1", "v2", "v3"]

n_rows = len(derainer_order)
n_cols = len(severity_order)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(5 * n_cols, 3.3 * n_rows),
    sharey=True,
)

if n_rows == 1:
    axes = np.array([axes])

for r, derainer in enumerate(derainer_order):
    for c, severity in enumerate(severity_order):
        ax = axes[r, c]

        df_sub = mseg_rain_instance_df[
            (mseg_rain_instance_df["derainer"] == derainer) &
            (mseg_rain_instance_df["severity"] == severity)
        ]

        data = [
            df_sub[df_sub["version"] == version]["miou_to_other_rain_instances"].dropna().values
            for version in version_order
        ]

        ax.boxplot(data, labels=version_order, showmeans=True)
        ax.grid(axis="y", alpha=0.3)

        if r == 0:
            ax.set_title(f"{severity.capitalize()} rain")

        if c == 0:
            ax.set_ylabel(f"{derainer}\nmIoU")
        else:
            ax.set_ylabel("")

        ax.set_xlabel("Version")

fig.suptitle(
    "MSeg: Do rain instances matter? Version-to-version mIoU agreement",
    y=1.01,
)
fig.tight_layout()

out_path = PLOT_DIR / "mseg_all_derainers_rain_instance_miou_grid.png"
fig.savefig(out_path, dpi=240, bbox_inches="tight")
plt.show()

print("Saved:", out_path)

#### Do Segmentors agree with each other

In [ ]:
from pathlib import Path

ROOT = Path("/path/to/project")

MSEG_CLEAN_DIR = ROOT / "mseg-semantic/temp_files/mseg-3m-48p_clean_aachen_universal_ss/360/gray"

print("MSeg clean folder:", MSEG_CLEAN_DIR)
print("Exists:", MSEG_CLEAN_DIR.exists())
print("Mask count:", len(list(MSEG_CLEAN_DIR.glob("*.png"))) if MSEG_CLEAN_DIR.exists() else 0)

In [ ]:
def list_mask_files(folder, exclude_cat=False, limit=None):
    folder = Path(folder)
    if not folder.exists():
        return []
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])
    if exclude_cat:
        files = [p for p in files if not p.name.startswith("cat_")]
    if limit is not None:
        files = files[:limit]
    return files

def read_mask(path):
    return np.array(Image.open(path))

def resize_mask_nearest(mask, target_shape):
    if mask.shape[:2] == target_shape:
        return mask
    img = Image.fromarray(mask.astype(np.uint8))
    img = img.resize((target_shape[1], target_shape[0]), resample=Image.NEAREST)
    return np.array(img)

def mask_miou(mask_a, mask_b, ignore_labels=(255,)):
    if mask_b.shape[:2] != mask_a.shape[:2]:
        mask_b = resize_mask_nearest(mask_b, mask_a.shape[:2])
    labels = np.union1d(np.unique(mask_a), np.unique(mask_b))
    labels = [int(x) for x in labels if int(x) not in ignore_labels]
    ious = []
    for label in labels:
        a = (mask_a == label)
        b = (mask_b == label)
        union = np.logical_or(a, b).sum()
        if union == 0:
            continue
        inter = np.logical_and(a, b).sum()
        ious.append(inter / union)

    return float(np.mean(ious)) if ious else np.nan


def parse_hf_input_name(name):
    m = re.match(
        r"^(?P<derainer>.+)_rain_(?P<severity>light|medium|heavy)_(?P<version>[123])$",
        name
    )
    if m is None:
        return None
    return {
        "derainer": m.group("derainer"),
        "severity": m.group("severity"),
        "version": f"v{m.group('version')}",
    }
def parse_mseg_key(key):
    m = re.match(
        r"^(?P<derainer>.+)_(?P<severity>light|medium|heavy)_(?P<version>v[123])$",
        key
    )
    if m is None:
        return None
    derainer = m.group("derainer")
    if derainer == "udrs2former":
        derainer = "udr"
    return {
        "derainer": derainer,
        "severity": m.group("severity"),
        "version": m.group("version"),
    }
def resolve_mseg_path(path_str):
    p = Path(path_str)
    if p.exists():
        return p
    return ROOT / path_str

In [ ]:
rows = []
mseg_clean_files = list_mask_files(MSEG_CLEAN_DIR, exclude_cat=True)
mseg_clean_map = {p.name: p for p in mseg_clean_files}
print("MSeg clean masks:", len(mseg_clean_map))
for key, rel_path in PRED_DIRS_ALL.items():
    meta = parse_mseg_key(key)
    if meta is None:
        continue
    derainer = meta["derainer"]
    severity = meta["severity"]
    version = meta["version"]
    pred_dir = resolve_mseg_path(rel_path)
    if derainer == "udr":
        pred_files = list_mask_files(pred_dir, exclude_cat=True, limit=50)
    else:
        pred_files = list_mask_files(pred_dir, exclude_cat=True)
    common_names = sorted(set(p.name for p in pred_files) & set(mseg_clean_map.keys()))
    for image_name in common_names:
        clean_mask = read_mask(mseg_clean_map[image_name])
        pred_mask = read_mask(pred_dir / image_name)
        miou = mask_miou(clean_mask, pred_mask)
        rows.append({
            "segmentor": "mseg",
            "derainer": derainer,
            "severity": severity,
            "version": version,
            "image_name": image_name,
            "miou_vs_clean": miou,
        })

for segmentor in HF_SEGMENTORS:
    clean_dir = HF_SEG_BASE / segmentor / "clean"
    clean_files = list_mask_files(clean_dir)
    clean_map = {p.name: p for p in clean_files}
    print(segmentor, "clean masks:", len(clean_map))
    seg_dir = HF_SEG_BASE / segmentor
    if not seg_dir.exists():
        continue
    for folder in sorted([p for p in seg_dir.iterdir() if p.is_dir()]):
        if folder.name == "clean":
            continue
        meta = parse_hf_input_name(folder.name)
        if meta is None:
            continue
        derainer = meta["derainer"]
        severity = meta["severity"]
        version = meta["version"]
        pred_files = list_mask_files(folder)
        pred_map = {p.name: p for p in pred_files}
        common_names = sorted(set(pred_map.keys()) & set(clean_map.keys()))
        for image_name in common_names:
            clean_mask = read_mask(clean_map[image_name])
            pred_mask = read_mask(pred_map[image_name])
            miou = mask_miou(clean_mask, pred_mask)
            rows.append({
                "segmentor": segmentor,
                "derainer": derainer,
                "severity": severity,
                "version": version,
                "image_name": image_name,
                "miou_vs_clean": miou,
            })
all4_df = pd.DataFrame(rows)
print("Total rows:", len(all4_df))
display(all4_df.head())
print("Counts per group:")
display(
    all4_df
    .groupby(["segmentor", "derainer", "severity", "version"])
    .size()
    .reset_index(name="n")
    .sort_values(["derainer", "severity", "version", "segmentor"])
)

In [ ]:
summary_df = (
    all4_df
    .groupby(["derainer", "severity", "segmentor"])
    .agg(
        n=("miou_vs_clean", "count"),
        mean_miou=("miou_vs_clean", "mean"),
        median_miou=("miou_vs_clean", "median"),
        std_miou=("miou_vs_clean", "std"),
        min_miou=("miou_vs_clean", "min"),
        max_miou=("miou_vs_clean", "max"),
    )
    .reset_index()
    .sort_values(["derainer", "severity", "segmentor"])
)
display(summary_df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

derainers_present = [d for d in DERAINER_ORDER if d in all4_df["derainer"].unique()]
n_rows = len(derainers_present)
n_cols = len(SEVERITY_ORDER)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(5.8 * n_cols, 4.4 * n_rows),
    sharey=True
)

if n_rows == 1:
    axes = np.array([axes])

for r, derainer in enumerate(derainers_present):
    for c, severity in enumerate(SEVERITY_ORDER):
        ax = axes[r, c]

        sub = all4_df[
            (all4_df["derainer"] == derainer) &
            (all4_df["severity"] == severity)
        ]

        data = [
            sub[sub["segmentor"] == seg]["miou_vs_clean"].dropna().values
            for seg in SEGMENTOR_ORDER
        ]

        ax.boxplot(data, labels=SEGMENTOR_ORDER, showmeans=True)
        ax.grid(axis="y", alpha=0.3)
        ax.set_ylim(0, 1.02)

        if r == 0:
            ax.set_title(f"{severity.capitalize()} rain", fontsize=12, pad=12)

        if c == 0:
            ax.set_ylabel(f"{derainer}\nmIoU vs clean", fontsize=11)
        else:
            ax.set_ylabel("")

        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=25, labelsize=9)

        counts = [len(x) for x in data]
        ax.text(
            0.02,
            0.96,
            f"n={counts}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.75, edgecolor="none")
        )

fig.suptitle(
    "All 4 segmentors | mIoU vs clean prediction",
    fontsize=15,
    y=0.995
)

fig.subplots_adjust(
    left=0.08,
    right=0.98,
    top=0.94,
    bottom=0.06,
    hspace=0.42,
    wspace=0.15
)

plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("/path/to/project")

PLOT_DIR = ROOT / "analysis_outputs/all4_segmentors_by_derainer"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

derainers_present = [d for d in DERAINER_ORDER if d in all4_df["derainer"].unique()]
severity_order = SEVERITY_ORDER
segmentor_order = SEGMENTOR_ORDER

saved_paths = []

for derainer in derainers_present:
    fig, axes = plt.subplots(
        1,
        len(severity_order),
        figsize=(17, 5),
        sharey=True
    )

    if len(severity_order) == 1:
        axes = [axes]

    for ax, severity in zip(axes, severity_order):
        sub = all4_df[
            (all4_df["derainer"] == derainer) &
            (all4_df["severity"] == severity)
        ]

        data = [
            sub[sub["segmentor"] == seg]["miou_vs_clean"].dropna().values
            for seg in segmentor_order
        ]

        counts = [len(x) for x in data]

        ax.boxplot(
            data,
            labels=segmentor_order,
            showmeans=True
        )

        ax.set_title(f"{severity.capitalize()} rain", fontsize=13, pad=12)
        ax.set_xlabel("Segmentor")
        ax.set_ylim(0, 1.02)
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", rotation=25, labelsize=9)

        ax.text(
            0.02,
            0.96,
            f"n={counts}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                alpha=0.75,
                edgecolor="none"
            )
        )

    axes[0].set_ylabel("mIoU vs clean prediction", fontsize=11)

    fig.suptitle(
        f"All 4 segmentors | Derainer: {derainer}",
        fontsize=15,
        y=1.03
    )

    fig.tight_layout()

    out_path = PLOT_DIR / f"all4_segmentors_miou_vs_clean_{derainer}.png"
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.show()

    saved_paths.append(out_path)
    print("Saved:", out_path)

print("\nSaved all derainer-specific plots:")
for p in saved_paths:
    print(p)

In [ ]:
!cd /path/to/project && tar -czf analysis_outputs.tar.gz analysis_outputs